In [1]:
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.nn import *
import torch
from tqdm import tqdm
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader
from constants import *
from model_gnn import GNN
import gc

In [2]:
el_df = pd.read_parquet('data/el_db.parquet')
el_df = el_df.loc[~el_df['allele'].str.contains(',')].reset_index(drop=True)

el_df['peptide'] = ['[CLS] ' + ' '.join(list(peptide)) + ' [SEP]' for peptide in el_df['peptide'].values]
unique_alleles = el_df['allele'].unique()
el_df

,peptide,el,source,allele
0,[CLS] A T S R T Y L Y R [SEP],1,netmhcpan_el,HLA-A*11:01
1,[CLS] V S K G T L V Q T K [SEP],1,netmhcpan_el,HLA-A*11:01
2,[CLS] Y L T K K F A E L [SEP],1,netmhcpan_el,HLA-A*02:07
3,[CLS] Y L F D L P L K V [SEP],1,netmhcpan_el,HLA-A*02:01
4,[CLS] S A V P K V M K V [SEP],1,netmhcpan_el,HLA-C*12:03
...,...,...,...,...
5036936,[CLS] D G Q A M L W D L N E G K H L Y T L D G ...,1,mhcmotifatlas,Gaga-BLB2*002:01
5036937,[CLS] K E P D V T I V S R L K D M V V F E G D ...,1,mhcmotifatlas,Gaga-BLB2*002:01
5036938,[CLS] A V V S A T E T L K M L N S N T E N P Y ...,1,mhcmotifatlas,Gaga-BLB2*002:01
5036939,[CLS] G S L L G D K T R M T E L S R D M N A Y ...,1,mhcmotifatlas,Gaga-BLB2*002:01


In [3]:
mhc_df = pd.read_parquet('data/mhc_db.parquet')
mhc_df = mhc_df.loc[mhc_df['allele'].isin(unique_alleles)]
el_df = el_df.loc[el_df['allele'].isin(mhc_df['allele'].values)]
unique_alleles = el_df['allele'].unique()
mhc_df['sequence'] = ['[CLS] ' + ' '.join(list(sequence)) + ' [SEP]' for sequence in mhc_df['sequence'].values]
mhc_df

,allele,sequence
171,BoLA-6*013:01,[CLS] M G P G T L F M L L L G A L V L I E T R ...
198,BoLA-DRB3*016:01,[CLS] M V C L Y F S G G S W M A A L I V M L M ...
1077,HLA-A*01:01,[CLS] M A V M A P R T L L L L L S G A L A L T ...
1079,HLA-A*01:03,[CLS] M A V M A P R T L L L L L S G A L A L T ...
1145,HLA-A*02:01,[CLS] M A V M A P R T L V L L L S G A L A L T ...
...,...,...
5476,HLA-E*01:01,[CLS] M V D G T L L L L L S E A L A L T Q T W ...
5477,HLA-E*01:03,[CLS] M V D G T L L L L L S E A L A L T Q T W ...
5525,HLA-G*01:01,[CLS] M V V M A P R T L F L L L S G A L T L T ...
5526,HLA-G*01:03,[CLS] M V V M A P R T L F L L L S G A L T L T ...


In [7]:
MAX_LEN_MHC = mhc_df['sequence'].str.split().str.len().max()
MAX_LEN_PEP = el_df['peptide'].drop_duplicates().str.split().str.len().max()
MAX_LEN_PEP, MAX_LEN_MHC

(np.int64(52), np.int64(374))

In [9]:
mhc_features = np.array([
    [TOKEN_VOCABULARY[e] for e in seq.split()] + [0 for _ in range(MAX_LEN_MHC - len(seq.split()))] for seq in mhc_df['sequence'].values
])

peptide_features = np.array([
    [TOKEN_VOCABULARY[e] for e in seq.split()] + [0 for _ in range(MAX_LEN_PEP - len(seq.split()))] for seq in el_df['peptide'].values
])

el_targets = np.expand_dims(el_df['el'].values, -1)

In [11]:
def get_hetero_data():
    data = []
    edge = torch.zeros((2, 1), dtype=torch.int32)
    for idx in tqdm(range(len(el_df))):
        entry = HeteroData()
        entry['peptide'].x = torch.tensor(peptide_features[idx], dtype=torch.int32).unsqueeze(0)
        entry['mhc'].x = torch.tensor(mhc_features[mhc_df['allele'] == el_df.iloc[idx]['allele']], dtype=torch.int32)
        entry['peptide', 'determines', 'mhc'].edge_index = edge
        entry['peptide', 'influences', 'peptide'].edge_index = edge
        entry['mhc', 'influences', 'mhc'].edge_index = edge
        entry['mhc'].y = torch.tensor(el_targets[idx], dtype=torch.float32).unsqueeze(-1)
        data.append(entry)
    return data


data = get_hetero_data()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 4673614/4673614 [33:32<00:00, 2322.57it/s]


## Training

In [ ]:
gc.collect()
torch.cuda.empty_cache()
   
EPOCHS = 30
TRAIN_BATCH_SIZE = 200

model = GNN(
    vocab_size=45,
    embedding_dim=128,
    dim_ff_enc_pep=128,
    n_heads_enc_pep=8,
    n_layers_enc_pep=6,
    dim_ff_enc_mhc=1024,
    n_heads_enc_mhc=8,
    n_layers_enc_mhc=6,
    n_heads_conv=8,
    n_layers_conv=2,
    dim_out_conv=32,
    dropout=0.1,
    act=F.leaky_relu,
    device1=DEVICE1,
    device2=DEVICE2,
)

state_dict = torch.load('weights/pretrain_mhc_ba.pt')
model.load_state_dict(state_dict, strict=True)

train_loader = DataLoader(data, batch_size=TRAIN_BATCH_SIZE, shuffle=True)

# freezing everything except convolution layers improves performance
for name, param in model.named_parameters():
    if param.requires_grad and not 'conv' in name:
        param.requires_grad = False

non_frozen_parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(non_frozen_parameters, lr=1e-4)

# Training loop
for epoch in range(1, EPOCHS+1):
    model.train()
    batches = tqdm(train_loader)
    bl = []
    for batch in batches:
        optimizer.zero_grad()
        out = model(batch)
        loss = F.binary_cross_entropy_with_logits(out, batch['mhc'].y.to(DEVICE2))
        bl.append(loss.item())
        loss.backward()
        optimizer.step()
        batches.set_description(f'Train epoch {epoch} - loss: {np.mean(bl[-100:]):.4f}')
        
torch.save(model.state_dict(), 'weights/pretrain_mhc_ba_el.pt')